Please edit the `ibmq_env.yaml` first, have it in same folder as this notebook, and then run the notebook.

In [ ]:
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import yaml
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import QiskitRuntimeService

BASE = Path.cwd()
CFG_PATH = BASE / 'ibmq_env.yaml'
if not CFG_PATH.exists():
    raise FileNotFoundError('Missing ibmq_env.yaml in this folder. Copy from ibmq_env.yaml.sample and fill values.')

def load_cfg(path: Path) -> dict:
    data = yaml.safe_load(path.read_text()) or {}
    return data.get('CONFIG', {}) if isinstance(data, dict) else {}

cfg = load_cfg(CFG_PATH)
IBMQ_API_KEY = (cfg.get('IBMQ_API_KEY') or os.environ.get('IBMQ_API_KEY') or os.environ.get('QISKIT_IBM_TOKEN') or '').strip()
IBMQ_INSTANCE = (cfg.get('IBMQ_SERVICE_CRN') or os.environ.get('IBMQ_SERVICE_CRN') or os.environ.get('QISKIT_IBM_INSTANCE') or '').strip()
IBMQ_CHANNEL = (cfg.get('IBMQ_CHANNEL') or os.environ.get('IBMQ_CHANNEL') or 'ibm_quantum_platform').strip()
USER_TAG = (cfg.get('USER_TAG') or os.environ.get('USER') or 'user').strip().replace(' ', '_')
OUTPUT_DIR = Path(cfg.get('OUTPUT_DIR') or './final_data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HTTP_PROXY = (cfg.get('HTTP_PROXY') or os.environ.get('HTTP_PROXY') or '').strip()
HTTPS_PROXY = (cfg.get('HTTPS_PROXY') or os.environ.get('HTTPS_PROXY') or '').strip()
if HTTP_PROXY:
    os.environ['HTTP_PROXY'] = HTTP_PROXY
if HTTPS_PROXY:
    os.environ['HTTPS_PROXY'] = HTTPS_PROXY

if not IBMQ_API_KEY or not IBMQ_INSTANCE:
    raise ValueError('IBMQ_API_KEY and IBMQ_SERVICE_CRN are required in ibmq_env.yaml.')

print('Config loaded for USER_TAG =', USER_TAG)
print('Output directory =', OUTPUT_DIR.resolve())

In [ ]:
def pick(d, *keys):
    for key in keys:
        if isinstance(d, dict) and key in d and d[key] is not None:
            return d[key]
    return None

def deep_pick(obj, keys_set):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in keys_set and v is not None:
                return v
            out = deep_pick(v, keys_set)
            if out is not None:
                return out
    elif isinstance(obj, list):
        for item in obj:
            out = deep_pick(item, keys_set)
            if out is not None:
                return out
    return None

def deep_path_pick(obj, path):
    cur = obj
    for p in path:
        if isinstance(cur, dict):
            if p not in cur:
                return None
            cur = cur[p]
        elif isinstance(cur, list):
            if not isinstance(p, int) or p < 0 or p >= len(cur):
                return None
            cur = cur[p]
        else:
            return None
    return cur

def to_dt(value):
    if value is None or value == '':
        return None
    if isinstance(value, (int, float)):
        if value > 10_000_000_000:
            value = value / 1000.0
        return datetime.fromtimestamp(value, tz=timezone.utc)
    text = str(value).strip().replace('Z', '+00:00')
    try:
        dt = datetime.fromisoformat(text)
        return dt if dt.tzinfo else dt.replace(tzinfo=timezone.utc)
    except ValueError:
        return None

def runtime_ms(start, end):
    start_dt = to_dt(start)
    end_dt = to_dt(end)
    if start_dt is None or end_dt is None:
        return None
    return (end_dt - start_dt).total_seconds() * 1000.0

def maybe_load_qasm_string(qasm_text):
    if not isinstance(qasm_text, str) or not qasm_text.strip():
        return None
    text = qasm_text.strip()
    try:
        if 'OPENQASM 3' in text.upper():
            from qiskit.qasm3 import loads as loads_qasm3
            return loads_qasm3(text)
        return QuantumCircuit.from_qasm_str(text)
    except Exception:
        return None

def find_qiskit_circuit(raw):
    qasm_text = (
        pick(raw, 'qasm', 'openqasm', 'qasm_str', 'qasm_string')
        or deep_pick(raw, {'qasm', 'openqasm', 'qasm_str', 'qasm_string'})
    )
    qc = maybe_load_qasm_string(qasm_text)
    if qc is not None:
        return qc

    pub_circ = (
        deep_path_pick(raw, ['params', 'pubs', 0, 'circuit'])
        or deep_path_pick(raw, ['inputs', 'pubs', 0, 'circuit'])
        or deep_path_pick(raw, ['job_detail', 'params', 'pubs', 0, 'circuit'])
    )
    if isinstance(pub_circ, str):
        return maybe_load_qasm_string(pub_circ)
    if isinstance(pub_circ, dict):
        for k in ['qasm', 'openqasm', 'qasm_str', 'qasm_string']:
            qc = maybe_load_qasm_string(pub_circ.get(k))
            if qc is not None:
                return qc
    if isinstance(pub_circ, QuantumCircuit):
        return pub_circ
    return None

def qiskit_detail_metrics(qc):
    clifford_ops = {
        'id', 'x', 'y', 'z', 'h', 's', 'sdg', 'sx', 'sxdg',
        'cx', 'cy', 'cz', 'swap', 'measure', 'barrier', 'reset'
    }
    op_names = []
    used_qubits = set()
    edges = set()
    for inst, qargs, _ in qc.data:
        name = str(inst.name).lower()
        op_names.append(name)
        idxs = []
        for q in qargs:
            qi = qc.find_bit(q).index
            idxs.append(qi)
            used_qubits.add(qi)
        if len(idxs) >= 2:
            for i in range(len(idxs)):
                for j in range(i + 1, len(idxs)):
                    a, b = sorted((idxs[i], idxs[j]))
                    edges.add((a, b))

    n_cliff = sum(1 for n in op_names if n in clifford_ops)
    n_non = len(op_names) - n_cliff

    if not used_qubits:
        n_comp = 0
    else:
        parent = {u: u for u in used_qubits}

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def union(a, b):
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[rb] = ra

        for a, b in edges:
            if a in parent and b in parent:
                union(a, b)

        n_comp = len({find(u) for u in used_qubits})

    return {
        'framework': 'qiskit',
        'critical_path_length': int(qc.depth()),
        'connected_components': int(n_comp),
        'num_cliffords': int(n_cliff),
        'num_non_cliffords': int(n_non),
        'num_parameters': int(len(qc.parameters)),
    }

def circuit_features_from_qc(qc):
    counts = qc.count_ops()
    one_q = 0
    two_q = 0
    three_q = 0
    for inst, qargs, _ in qc.data:
        if len(qargs) == 1:
            one_q += 1
        elif len(qargs) == 2:
            two_q += 1
        elif len(qargs) >= 3:
            three_q += 1

    detail = qiskit_detail_metrics(qc)
    return {
        'input_format': 'qasm',
        'n_qubits': int(qc.num_qubits),
        'depth': int(qc.depth()),
        'n_ops': int(qc.size()),
        'single_qubit_gates': one_q,
        'two_qubit_gates': two_q,
        'three_qubit_gates': three_q,
        'measure_ops': int(counts.get('measure', 0)),
        'gate_counts_json': json.dumps({str(k): int(v) for k, v in counts.items()}),
        'framework': detail.get('framework'),
        'critical_path_length': detail.get('critical_path_length'),
        'connected_components': detail.get('connected_components'),
        'num_cliffords': detail.get('num_cliffords'),
        'num_non_cliffords': detail.get('num_non_cliffords'),
        'num_parameters': detail.get('num_parameters'),
        'circuit_metrics_json': json.dumps(detail),
    }

def _coerce_qc(obj):
    if isinstance(obj, QuantumCircuit):
        return obj
    if isinstance(obj, str):
        return maybe_load_qasm_string(obj)
    if isinstance(obj, dict):
        for k in ['qasm', 'openqasm', 'qasm_str', 'qasm_string']:
            qc = maybe_load_qasm_string(obj.get(k))
            if qc is not None:
                return qc
    return None

def runtime_job_circuit_features(service, job_id, backend_name=None):
    if not job_id:
        return {}
    try:
        job = service.job(job_id)
        inputs = job.inputs
        cands = []
        if isinstance(inputs, dict):
            circs = inputs.get('circuits')
            if isinstance(circs, list):
                cands.extend(circs)
            elif circs is not None:
                cands.append(circs)
            pubs = inputs.get('pubs')
            if isinstance(pubs, list):
                for pub in pubs:
                    if isinstance(pub, dict) and 'circuit' in pub:
                        cands.append(pub['circuit'])
                    elif isinstance(pub, (list, tuple)) and len(pub) > 0:
                        cands.append(pub[0])

        for c in cands:
            qc = _coerce_qc(c)
            if qc is None:
                continue
            if backend_name:
                try:
                    backend_obj = service.backend(backend_name)
                    qc = transpile(qc, backend=backend_obj, optimization_level=1)
                except Exception:
                    pass
            out = circuit_features_from_qc(qc)
            out['input_format'] = 'runtime_inputs'
            return out
    except Exception:
        return {}
    return {}

In [ ]:
service = QiskitRuntimeService(
    channel=IBMQ_CHANNEL,
    token=IBMQ_API_KEY,
    instance=IBMQ_INSTANCE,
)

limit = 200
page_size = 25
max_retries = 5
skip = 0
summaries = []

while len(summaries) < limit:
    batch_size = min(page_size, limit - len(summaries))
    ok = False
    for attempt in range(max_retries):
        try:
            resp = service._active_api_client.jobs_get(limit=batch_size, skip=skip, descending=True)
            ok = True
            break
        except Exception as e:
            msg = str(e)
            if '503' in msg or 'Service Unavailable' in msg:
                time.sleep(2 ** attempt)
                continue
            raise
    if not ok:
        break

    jobs = resp.get('jobs', [])
    if not jobs:
        break
    summaries.extend(jobs)
    skip += len(jobs)

rows = []
for summary in summaries:
    job_id = pick(summary, 'id', 'job_id')
    if not job_id:
        continue

    detail = {}
    meta = {}
    for attempt in range(max_retries):
        try:
            detail = service._active_api_client.job_get(job_id=job_id, exclude_params=False) or {}
            break
        except Exception as e:
            if '503' in str(e):
                time.sleep(2 ** attempt)
                continue
            break
    for attempt in range(max_retries):
        try:
            meta = service._active_api_client.job_metadata(job_id=job_id) or {}
            break
        except Exception as e:
            if '503' in str(e):
                time.sleep(2 ** attempt)
                continue
            break

    raw = dict(summary)
    raw['job_detail'] = detail
    raw['job_metadata'] = meta
    for k, v in detail.items():
        if k not in raw or raw[k] in (None, '', {}):
            raw[k] = v

    start = (
        pick(raw, 'started', 'started_at', 'running_at')
        or deep_path_pick(raw, ['state', 'timestamps', 'running'])
        or deep_path_pick(raw, ['job_metadata', 'timestamps', 'running'])
        or deep_pick(raw, {'running', 'running_at', 'started_at', 'start_time'})
    )
    end = (
        pick(raw, 'completed', 'completed_at', 'ended_at', 'finished_at')
        or deep_path_pick(raw, ['state', 'timestamps', 'finished'])
        or deep_path_pick(raw, ['job_metadata', 'timestamps', 'finished'])
        or deep_pick(raw, {'finished', 'finished_at', 'ended_at', 'completed_at'})
    )

    run_ms = runtime_ms(start, end)
    if run_ms is None:
        sec = (
            pick(raw, 'estimated_running_time_seconds')
            or deep_path_pick(raw, ['job_detail', 'estimated_running_time_seconds'])
            or deep_path_pick(raw, ['job_metadata', 'usage', 'quantum_seconds'])
            or deep_pick(raw, {'estimated_running_time_seconds', 'quantum_seconds'})
        )
        if isinstance(sec, (int, float)):
            run_ms = float(sec) * 1000.0

    shots = (
        pick(raw, 'shots')
        or deep_path_pick(raw, ['params', 'shots'])
        or deep_path_pick(raw, ['params', 'pubs', 0, 'shots'])
        or deep_path_pick(raw, ['inputs', 'run_options', 'shots'])
        or deep_pick(raw, {'shots', 'num_shots'})
    )

    backend_name = pick(raw, 'backend') or deep_pick(raw, {'backend', 'backend_name', 'target'})
    feats = {}

    qc = find_qiskit_circuit(raw)
    if qc is not None:
        feats = circuit_features_from_qc(qc)

    if not feats or feats.get('n_qubits') is None:
        extra = runtime_job_circuit_features(service, job_id=job_id, backend_name=backend_name)
        if extra:
            feats.update(extra)

    rows.append({
        'provider': 'ibmq',
        'job_id': job_id,
        'backend': backend_name,
        'status': pick(raw, 'status', 'state') or deep_path_pick(raw, ['state', 'status']) or deep_pick(raw, {'status', 'state'}),
        'created_at': pick(raw, 'created', 'created_at') or deep_pick(raw, {'created', 'created_at', 'creation_date'}),
        'started_at': start,
        'ended_at': end,
        'runtime_ms': run_ms,
        'shots': shots,
        'session_id': pick(raw, 'session_id') or deep_pick(raw, {'session_id'}),
        'input_format': feats.get('input_format'),
        'n_qubits': feats.get('n_qubits'),
        'depth': feats.get('depth'),
        'n_ops': feats.get('n_ops'),
        'single_qubit_gates': feats.get('single_qubit_gates'),
        'two_qubit_gates': feats.get('two_qubit_gates'),
        'three_qubit_gates': feats.get('three_qubit_gates'),
        'measure_ops': feats.get('measure_ops'),
        'gate_counts_json': feats.get('gate_counts_json'),
        'qiskit_framework': feats.get('framework'),
        'critical_path_length': feats.get('critical_path_length'),
        'connected_components': feats.get('connected_components'),
        'num_cliffords': feats.get('num_cliffords'),
        'num_non_cliffords': feats.get('num_non_cliffords'),
        'num_parameters': feats.get('num_parameters'),
        'circuit_metrics_json': feats.get('circuit_metrics_json'),
    })

df = pd.DataFrame(rows)
for col in ['created_at', 'started_at', 'ended_at']:
    df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
df['runtime_ms'] = pd.to_numeric(df['runtime_ms'], errors='coerce')
df['shots'] = pd.to_numeric(df['shots'], errors='coerce')

out_path = OUTPUT_DIR / f'{USER_TAG}_data.csv'
df.to_csv(out_path, index=False)

print('Fetched summaries:', len(summaries))
print('Saved:', out_path)
print('Rows:', len(df))
print('Rows with runtime_ms:', int(df['runtime_ms'].notna().sum()) if 'runtime_ms' in df else 0)
print('Rows with circuit metrics (n_qubits):', int(df['n_qubits'].notna().sum()) if 'n_qubits' in df else 0)
print('Rows with Qiskit detail metrics:', int(df['critical_path_length'].notna().sum()) if 'critical_path_length' in df else 0)
df.head()